# **Training the segmenter**

This notebooks presents how to train an automatic segmenter based on BERT AutoModelForTokenClassfication model, in order to automatically segment medieval texts.



# 1. Libraries import


Install some libraries:

In [ ]:
!pip install evaluate
!pip install langid
!pip install numpyencoder

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 33.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langid: filename=langid-1.1.6-py3-none-any.whl size=1941171 sha256=cd130c7632c34a9219e29713c94a2bc90dac790bf934d841a924ed22824acbb1
  Stored in directory: /root/.cache/pip/wheels/32/6a/b6/b7eb43a6ad55b139c15c5daa29f3707659cfa6944d3c696f5b
Successfully built langid


Clone the repository of the workshop (a fork of Aquilign):

In [ ]:
!git clone https://github.com/ProMeText/multilingual-medieval-aligner-workshop.git

Cloning into 'multilingual-medieval-aligner-workshop'...
remote: Enumerating objects: 6793, done.
remote: Counting objects: 100% (980/980), done.
remote: Compressing objects: 100% (406/406), done.
remote: Total 6793 (delta 628), reused 610 (delta 563), pack-reused 5813 (from 1)
Receiving objects: 100% (6793/6793), 365.05 MiB | 17.70 MiB/s, done.
Resolving deltas: 100% (2926/2926), done.


Add path to the system and init.py:

In [ ]:
import sys
sys.path.append('/content/multilingual-medieval-aligner-workshop')

!touch /content/multilingual-medieval-aligner-workshop/aquilign/__init__.py
!touch /content/multilingual-medieval-aligner-workshop/aquilign/align/__init__.py
!touch /content/multilingual-medieval-aligner-workshop/aquilign/preproc/__init__.py

**Import** the libraries and the functions:

In [ ]:
from transformers import BertTokenizer, Trainer, TrainingArguments, AutoModelForTokenClassification, set_seed, TrainerCallback, EarlyStoppingCallback
import aquilign.preproc.tok_trainer_functions as trainer_functions
import aquilign.preproc.eval as evaluation
import aquilign.preproc.utils as utils
import re
import os
import json
import glob
import argparse
import jsonschema

Make the repo folder as root folder:

In [ ]:
os.chdir('/content/multilingual-medieval-aligner-workshop')

# 2. Main function


In [ ]:
# Callback to save every N epoch (usefull for small datasets)
class SaveEveryNEpochsCallback(TrainerCallback):
    def __init__(self, save_every):
        self.save_every = save_every

    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch % self.save_every == 0:
            control.should_save = True  # Forces saving
        else:
            control.should_save = False  # Skips saving

In [ ]:
def training_trainer(modelName,
                     train_dataset,
                     dev_dataset,
                     eval_dataset,
                     num_train_epochs,
                     batch_size,
                     logging_steps,
                     use_cpu,
                     bf_16,
                     out_name,
                     save_every,
                     early_stopping,
                     keep_punct=True):

    train_lines = utils.json_corpus_to_lines(train_dataset, keep_punct)
    dev_lines = utils.json_corpus_to_lines(dev_dataset, keep_punct)
    eval_lines, delimiter = utils.json_corpus_to_lines(eval_dataset, keep_punct, return_delimiter=True)
    eval_data_lang = eval_dataset.split("/")[-2]

    model = AutoModelForTokenClassification.from_pretrained(modelName, num_labels=3)
    tokenizer = BertTokenizer.from_pretrained(modelName, max_length=10)

    # Train corpus
    print("Train corpus preparation")
    train_texts_and_labels = utils.convertToSubWordsSentencesAndLabels(train_lines, tokenizer=tokenizer, delimiter=delimiter)
    train_dataset = trainer_functions.SentenceBoundaryDataset(train_texts_and_labels, tokenizer)

    # Dev corpus
    print("Dev corpus preparation")
    dev_texts_and_labels = utils.convertToSubWordsSentencesAndLabels(dev_lines, tokenizer=tokenizer, delimiter=delimiter)
    dev_dataset = trainer_functions.SentenceBoundaryDataset(dev_texts_and_labels, tokenizer)

    if '/' in modelName:
        name_of_model = re.split('/', modelName)[1]
    else:
        name_of_model = modelName

    # training arguments
    # num train epochs, logging_steps and batch_size should be provided
    # evaluation is done by epoch and the best model of each one is stored in a folder "results_+name"
    training_args = TrainingArguments(
        output_dir=f"results_{out_name}/epoch{num_train_epochs}_bs{batch_size}",
        num_train_epochs=num_train_epochs,
        logging_strategy="epoch",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        #specific change for the notebook
        dataloader_num_workers=2,
        ##
        dataloader_prefetch_factor=4,
        bf16=bf_16,
        #specific addition for the notebook
        report_to="none",
        ##
        use_cpu=use_cpu,
        save_strategy="epoch",
        load_best_model_at_end=True
        # best model is evaluated on loss
    )

    # define the trainer : model, training args, datasets and the specific compute_metrics defined in functions file
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=trainer_functions.compute_metrics,
        callbacks=[SaveEveryNEpochsCallback(save_every=save_every),
                   EarlyStoppingCallback(early_stopping_patience=early_stopping)]
    )

    print("Evaluating model before finetuning.")
    eval_results = evaluation.run_eval(data=eval_lines,
                                       model_path=modelName,
                                       tokenizer_name=modelName,
                                       verbose=False,
                                       delimiter=delimiter)

    # fine-tune the model
    print("Starting training")
    trainer.train()
    print("End of training")

    # get the best model path
    best_model_path = trainer.state.best_model_checkpoint
    print(f"Evaluation.")

    # print the whole log_history with the compute metrics
    best_precision_step, best_step_metrics = utils.get_best_step(trainer.state.log_history)

    all_checkpoints = glob.glob(f"results_{out_name}/epoch{num_train_epochs}_bs{batch_size}/checkpoint-*")
    as_ints = [int(checkpoint.replace(f"results_{out_name}/epoch{num_train_epochs}_bs{batch_size}/checkpoint-", ""))
               for checkpoint in all_checkpoints]

    all_diffs = [abs(best_precision_step - checkpoint) for checkpoint in as_ints]
    min_index = all_diffs.index(min(all_diffs))
    best_model_path = all_checkpoints[min_index]

    print(f"Best model path according to recall: {best_model_path}")
    print(f"Full metrics: {best_step_metrics}")

    eval_results = evaluation.run_eval(data=eval_lines,
                        model_path=best_model_path,
                        tokenizer_name=tokenizer.name_or_path,
                        verbose=False,
                        delimiter=delimiter)

    # We move the best state dir name to "best"
    new_best_path = f"results_{out_name}/epoch{num_train_epochs}_bs{batch_size}/best"
    try:
        os.rmdir(new_best_path)
    except FileNotFoundError:
        pass
    os.rename(best_model_path, new_best_path)

    with open(f"{new_best_path}/model_name", "w") as model_name:
        model_name.write(modelName)

    with open(f"{new_best_path}/eval.txt", "w") as evaluation_results:
        evaluation_results.write(eval_results)

    with open(f"{new_best_path}/metrics.json", "w") as metrics:
        json.dump(best_step_metrics, metrics)

    print(f"\n\nBest model can be found at : {new_best_path} ")
    print(f"You should remove the following directories by using `rm -r results_{out_name}/epoch{num_train_epochs}_bs{batch_size}/checkpoint-*`")

    # functions returns best model_path
    return new_best_path

# 3. Implementation and training


## Arguments


- the **model** we want to train, in our case a the google-bert multilingual model (designed for contemporary languages):

In [ ]:
model = 'google-bert/bert-base-multilingual-cased'

- the **datasets** for training : **train**, **dev** and **eval** sets:

In [ ]:
train_dataset= 'data/to-train/multilingual/train.json'
dev_dataset = 'data/to-train/multilingual/dev.json'
eval_dataset = 'data/to-train/multilingual/test.json'

**Note**: the sets don't correspond to the real sets which were used for training. Indeed, they have been reduced in order to perform a simulation of training in reasonable times for a workshop. Cf.  ProMeText/Multilingual_Aegidius/data/segmentation_data/split/multilingual for real training split sets.

The training data must follow the following structure and will be validated against a specific JSON schema. Example:

`{"metadata": {"name": "test", "delimiter": "\u00a3", "examples_number": "", "chars": "", "words": "", "langs": ["en", "fr", "pt", "es", "it", "ca", "la"]}, "examples": [{"example": "u de peisuns, \u00a3derechief refurmerunt en la resurrectiun \u00a3que uns chevel d'els perirat. D. :\u00a3Se li chevel est trenchiez u li ungle \u00a3e si il repairent en lur lungur, \u00a3dunt ne serunt il lait? M. : \u00a3Nen est pas \u00e0 entendre \u00a3que il repairent en lur premerain liu, \u00a3mes, cum potiers fraint", "lang": "fr"}, {"example": "menatos. \u00a3Allor domanda monsignor Galvano alla damigella \u00a3perch\u2019elli mena cavaliere \u00a3che va", "lang": "it"}]
}`

- the **number of epochs** we want to train the model on (here, for the demo, 2, can be 100):

In [ ]:
num_train_epochs = 2

- the **batch size** (here, for, the demo, 8, can be 128):

In [ ]:
batch_size = 8

- the **device**:

In [ ]:
use_cpu = False

- bf16 mixed precision is used:

In [ ]:
bf_16 = True

- the **name** of the trained model:

In [ ]:
out_name = 'multilingual_model'

- the **frequency** at which the **model should be saved**:

In [ ]:
save_every = 2

- early stopping:

In [ ]:
early_stopping = 10

- **logging steps**:

In [ ]:
logging_steps = 10

## Let's train the model !

In [ ]:
## 5 min
training_trainer(model,
                     train_dataset,
                     dev_dataset,
                     eval_dataset,
                     num_train_epochs,
                     batch_size,
                     logging_steps,
                     use_cpu,
                     bf_16,
                     out_name,
                     save_every,
                     early_stopping)

Test on data/to-train/multilingual/train.json passed.
Test on data/to-train/multilingual/dev.json passed.
Test on data/to-train/multilingual/test.json passed.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Performing regexp based tokenization evaluation


(0.96849887495982, [0.9019376579612468, 0.6360360360360361, 1.0], [0.9636363636363636, 0.3775401069518717, 1.0], [0.9317667536988686, 0.4738255033557047, 1.0])
Performing bert-based tokenization evaluation
|           | Synt (None, Delim.)                           | Bert (None, Delim., Pad.)                                     |
|-----------+-----------------------------------------------+---------------------------------------------------------------|
| Accuracy  | 0.96849887495982                              | 0.8088235294117647                                            |
| Precision | [0.9019376579612468, 0.6360360360360361, 1.0] | [0.8320503848845346, 0.16232412714955707, 0.9318855878952396] |
| Recall    | [0.9636363636363636, 0.3775401069518717, 1.0] | [0.21404140414041403, 0.6663101604278074, 0.9929298339582218] |
| F1-score  | [0.9317667536988686, 0.4738255033557047, 1.0] | [0.3404925544100802, 0.26105174942384246, 0.9614397220133288] |
Starting training


Epoch,Training Loss,Validation Loss,Accurracy,Recall,Precision,F1
1,0.055500,0.022233,{'accuracy': 0.9915137285826942},"[0.9836514522821577, 0.7792415169660679, 1.0]","[0.9772043365348942, 0.833119931711481, 0.9999647017296153]","[0.9804172956430034, 0.8052805280528053, 0.9999823505533102]"
2,0.013000,0.020077,{'accuracy': 0.9925442684063374},"[0.9828630705394191, 0.8327345309381238, 1.0]","[0.9826184352443376, 0.834733893557423, 1.0]","[0.9827407376675102, 0.8337330135891287, 1.0]"


Starting eval
Eval finished
Starting eval
Eval finished
End of training
Evaluation.
[{'loss': 0.0555, 'grad_norm': 0.20298956334590912, 'learning_rate': 2.5257731958762887e-05, 'epoch': 1.0, 'step': 97}, {'eval_loss': 0.02223268896341324, 'eval_accurracy': {'accuracy': 0.9915137285826942}, 'eval_recall': [0.9836514522821577, 0.7792415169660679, 1.0], 'eval_precision': [0.9772043365348942, 0.833119931711481, 0.9999647017296153], 'eval_f1': [0.9804172956430034, 0.8052805280528053, 0.9999823505533102], 'eval_runtime': 14.9474, 'eval_samples_per_second': 25.222, 'eval_steps_per_second': 3.211, 'epoch': 1.0, 'step': 97}, {'loss': 0.013, 'grad_norm': 0.23535305261611938, 'learning_rate': 2.577319587628866e-07, 'epoch': 2.0, 'step': 194}, {'eval_loss': 0.02007659152150154, 'eval_accurracy': {'accuracy': 0.9925442684063374}, 'eval_recall': [0.9828630705394191, 0.8327345309381238, 1.0], 'eval_precision': [0.9826184352443376, 0.834733893557423, 1.0], 'eval_f1': [0.9827407376675102, 0.83373301358

'results_multilingual_model/epoch2_bs8/best'